In [1]:
class GinzburgCohomologyCalculator:
    """
    Ginzburg dg代数のコホモロジーを計算・探索するための統合クラス
    カスタムポテンシャルの指定と、自動生成に対応しました。
    """
    def __init__(self, a, b, c, custom_W=None, potential_type='generic', seed=None):
        self.a = a
        self.b = b
        self.c = c
        self.elements = {}
        self.x_ids, self.y_ids, self.z_ids = [], [], []
        self.x_s_ids, self.y_s_ids, self.z_s_ids = [], [], []
        self.t_ids = []
        self.basis_cache = {}
        
        # Quiverの構造構築
        self._build_quiver()
        
        # ポテンシャル W の設定
        # custom_W が渡された場合はそれを採用し、無ければ自動生成する
        if custom_W is not None:
            self.W_coeffs = custom_W
            self.potential_type = 'custom'
        else:
            self.potential_type = potential_type
            self.W_coeffs = self.generate_potential(potential_type, seed)

    def _build_quiver(self):
        """Quiverの構造を一度だけ構築する"""
        el_idx = 0
        a, b, c = self.a, self.b, self.c
        
        for _ in range(a): self.elements[el_idx] = {'s': 1, 'e': 2, 'deg': 0, 'len': 1}; self.x_ids.append(el_idx); el_idx += 1
        for _ in range(b): self.elements[el_idx] = {'s': 2, 'e': 3, 'deg': 0, 'len': 1}; self.y_ids.append(el_idx); el_idx += 1
        for _ in range(c): self.elements[el_idx] = {'s': 3, 'e': 1, 'deg': 0, 'len': 1}; self.z_ids.append(el_idx); el_idx += 1
            
        for _ in range(a): self.elements[el_idx] = {'s': 2, 'e': 1, 'deg': -1, 'len': 2}; self.x_s_ids.append(el_idx); el_idx += 1
        for _ in range(b): self.elements[el_idx] = {'s': 3, 'e': 2, 'deg': -1, 'len': 2}; self.y_s_ids.append(el_idx); el_idx += 1
        for _ in range(c): self.elements[el_idx] = {'s': 1, 'e': 3, 'deg': -1, 'len': 2}; self.z_s_ids.append(el_idx); el_idx += 1

        for v in [1, 2, 3]: self.elements[el_idx] = {'s': v, 'e': v, 'deg': -2, 'len': 3}; self.t_ids.append(el_idx); el_idx += 1

    def generate_potential(self, potential_type='generic', seed=None):
        """
        ポテンシャル W を自動生成するメソッド。
        任意の (a,b,c) に対して、規則的で特別な形のポテンシャルを生成できるように一般化しました。
        """
        a, b, c = self.a, self.b, self.c
        C = {}
        # 各変数の共通して取れるインデックスの最大範囲
        N = min(a, b, c) 
        
        if potential_type == 'generic':
            if seed is not None:
                set_random_seed(seed)
            for i in range(a):
                for j in range(b):
                    for k in range(c):
                        C[(i, j, k)] = ZZ.random_element(-100, 101)
                        
        elif potential_type == 'diagonal':
            # 対角項のみを足し合わせる (例: x_0 y_0 z_0 + x_1 y_1 z_1 + ...)
            for i in range(N):
                C[(i, i, i)] = 1
                
        elif potential_type == 'dense_ones':
            # 考えうる全ての項の係数を 1 にする (ベタ塗り)
            for i in range(a):
                for j in range(b):
                    for k in range(c):
                        C[(i, j, k)] = 1
                        
        elif potential_type == 'alternating':
            # インデックスの和の偶奇で符号が変わる市松模様的なポテンシャル
            for i in range(a):
                for j in range(b):
                    for k in range(c):
                        C[(i, j, k)] = 1 if (i + j + k) % 2 == 0 else -1
                        
        elif potential_type == 'levi_civita':
            # 相異なるインデックスの組に対し、置換の符号(偶置換=1, 奇置換=-1)を係数とする
            # (旧 symmetric の完全な一般化)
            import itertools
            if N >= 3:
                # 0 から N-1 までの3つの相異なるインデックスを選ぶ
                for indices in itertools.permutations(range(N), 3):
                    i, j, k = indices
                    # 転倒数(inversions)を計算して偶奇を判定
                    inversions = 0
                    if i > j: inversions += 1
                    if i > k: inversions += 1
                    if j > k: inversions += 1
                    C[(i, j, k)] = 1 if inversions % 2 == 0 else -1
            else:
                # N < 3 の場合は反対称テンソルが作れないため diagonal にフォールバック
                for i in range(N):
                    C[(i, i, i)] = 1
                    
        elif potential_type == 'cyclic_shift':
            # 対角項を -1 とし、インデックスが巡回する項を +1 とする特異な形
            # (旧 singular の一般化)
            for i in range(N):
                C[(i, i, i)] = -1
            if N >= 2:
                for i in range(N):
                    # インデックスを1つずつずらす
                    j = (i + 1) % N
                    k = (i + 2) % N if N >= 3 else (i + 1) % N
                    C[(i, j, k)] = 1
                    
        return C

    def display_potential(self):
        """現在設定されているポテンシャル W を数式として出力する"""
        terms = []
        for i in range(self.a):
            for j in range(self.b):
                for k in range(self.c):
                    # 辞書に存在しないキーは係数0として扱う
                    coeff = self.W_coeffs.get((i, j, k), 0)
                    if coeff == 0:
                        continue
                    
                    term = f"x_{i} y_{j} z_{k}"
                    if coeff == 1:
                        terms.append(f"+ {term}")
                    elif coeff == -1:
                        terms.append(f"- {term}")
                    elif coeff > 0:
                        terms.append(f"+ {coeff}{term}")
                    else:
                        terms.append(f"- {abs(coeff)}{term}")
                        
        if not terms:
            print("W = 0")
        else:
            expr = " ".join(terms)
            if expr.startswith("+ "):
                expr = expr[2:] # 先頭の余分な + を削除
            print(f"W = {expr}")

    def get_basis(self, target_deg, target_len):
        """DFSによる基底探索 (キャッシュ付き)"""
        cache_key = (target_deg, target_len)
        if cache_key in self.basis_cache:
            return self.basis_cache[cache_key]
            
        basis = []
        def dfs(curr_node, curr_deg, curr_len, path):
            if curr_len == target_len:
                if curr_deg == target_deg: basis.append(tuple(path))
                return
            for el_id, el in self.elements.items():
                if el['s'] == curr_node and curr_deg + el['deg'] >= target_deg and curr_len + el['len'] <= target_len:
                    dfs(el['e'], curr_deg + el['deg'], curr_len + el['len'], path + [el_id])
                    
        for start in [1, 2, 3]: dfs(start, 0, 0, [])
        self.basis_cache[cache_key] = basis
        return basis

    def _get_derivations(self, p):
        """保持しているポテンシャル W から素数体 GF(p) 上での導分を計算"""
        F = GF(p)
        a, b, c = self.a, self.b, self.c
        
        # 整数の係数を GF(p) の要素に変換 (辞書にない項は0)
        C_F = {}
        for i in range(a):
            for j in range(b):
                for k in range(c):
                    val = self.W_coeffs.get((i, j, k), 0)
                    C_F[(i, j, k)] = F(val)

        derivations = {el_id: [] for el_id in self.elements}
        for i in range(a): derivations[self.x_s_ids[i]] = [(C_F[(i, j, k)], [self.y_ids[j], self.z_ids[k]]) for j in range(b) for k in range(c) if C_F[(i, j, k)] != 0]
        for j in range(b): derivations[self.y_s_ids[j]] = [(C_F[(i, j, k)], [self.z_ids[k], self.x_ids[i]]) for i in range(a) for k in range(c) if C_F[(i, j, k)] != 0]
        for k in range(c): derivations[self.z_s_ids[k]] = [(C_F[(i, j, k)], [self.x_ids[i], self.y_ids[j]]) for i in range(a) for j in range(b) if C_F[(i, j, k)] != 0]

        derivations[self.t_ids[0]] = [(F(1), [self.x_ids[i], self.x_s_ids[i]]) for i in range(a)] + [(F(-1), [self.z_s_ids[k], self.z_ids[k]]) for k in range(c)]
        derivations[self.t_ids[1]] = [(F(1), [self.y_ids[j], self.y_s_ids[j]]) for j in range(b)] + [(F(-1), [self.x_s_ids[i], self.x_ids[i]]) for i in range(a)]
        derivations[self.t_ids[2]] = [(F(1), [self.z_ids[k], self.z_s_ids[k]]) for k in range(c)] + [(F(-1), [self.y_s_ids[j], self.y_ids[j]]) for j in range(b)]

        return derivations

    def _diff_path(self, path, derivations):
        """パスの微分を計算"""
        res = []
        sign = 1
        for i, el_id in enumerate(path):
            for c_val, term_path in derivations[el_id]:
                res.append((c_val * sign, tuple(path[:i]) + tuple(term_path) + tuple(path[i+1:])))
            if self.elements[el_id]['deg'] % 2 != 0: sign *= -1
        return res

    def _calculate_over_prime(self, p, target_k, L):
        """コア計算エンジン: 疎行列(Sparse Matrix)を用いてランクを計算"""
        F = GF(p)
        V_prev = self.get_basis(-target_k + 1, L)
        V_curr = self.get_basis(-target_k, L)
        V_next = self.get_basis(-target_k - 1, L)
        
        if len(V_curr) == 0:
            return 0, 0, 0, len(V_prev), len(V_curr), len(V_next)

        V_prev_map = {path: i for i, path in enumerate(V_prev)}
        V_curr_map = {path: i for i, path in enumerate(V_curr)}

        derivations = self._get_derivations(p)

        # 行列 D_{-k} の計算
        curr_entries = {}
        for j, path in enumerate(V_curr):
            for c_val, res_path in self._diff_path(path, derivations):
                if res_path in V_prev_map:
                    r = V_prev_map[res_path]
                    curr_entries[(r, j)] = curr_entries.get((r, j), F(0)) + c_val
                    
        mat_curr = matrix(F, len(V_prev), len(V_curr), curr_entries, sparse=True)
        rank_curr = mat_curr.rank()
        dim_ker = len(V_curr) - rank_curr

        # 行列 D_{-(k+1)} の計算
        rank_next = 0
        if len(V_next) > 0:
            next_entries = {}
            for j, path in enumerate(V_next):
                for c_val, res_path in self._diff_path(path, derivations):
                    if res_path in V_curr_map:
                        r = V_curr_map[res_path]
                        next_entries[(r, j)] = next_entries.get((r, j), F(0)) + c_val
            mat_next = matrix(F, len(V_curr), len(V_next), next_entries, sparse=True)
            rank_next = mat_next.rank()

        dim_H = dim_ker - rank_next
        return dim_H, rank_curr, rank_next, len(V_prev), len(V_curr), len(V_next)

    def compute_single_cohomology(self, target_k, L, verbose=True):
        """特定のコホモロジー H^{-k}_L を計算"""
        dim_1, r_c1, r_n1, v_prev, v_curr, v_next = self._calculate_over_prime(32003, target_k, L)
        
        if verbose:
            print(f"--- Computing H^{{-{target_k}}} at Length L={L} for Q({self.a},{self.b},{self.c}) [{self.potential_type.upper()}] ---")
            print(f"Dimensions: V_{{-{target_k-1}}}={v_prev}, V_{{-{target_k}}}={v_curr}, V_{{-{target_k+1}}}={v_next}")
            
        if dim_1 == 0:
            if verbose:
                print(f"rank(D_{{-{target_k}}}) = {r_c1}")
                print(f"rank(D_{{-{target_k+1}}}) = {r_n1}")
                print(f"--> dim H^{{-{target_k}}}_L = 0  (Mathematically Guaranteed over Q!)\n")
            return 0

        dim_2, _, _, _, _, _ = self._calculate_over_prime(31991, target_k, L)
        final_dim = min(dim_1, dim_2)

        if verbose:
            print(f"rank(D_{{-{target_k}}}) = {r_c1}")
            print(f"rank(D_{{-{target_k+1}}}) = {r_n1}")
            if dim_1 == dim_2:
                print(f"--> Double-check Passed! dim H^{{-{target_k}}}_L = {final_dim} (Confirmed)\n")
            else:
                print(f"--> Characteristic Dependent! (GF(32003): {dim_1}, GF(31991): {dim_2})")
                print(f"--> dim H^{{-{target_k}}}_L = {final_dim} (Adopted smaller dim)\n")
                
        return final_dim

    def generate_collapse_map(self, max_k=4, max_L=8):
        """H^{-k}_L の一覧表（Collapse Map）を生成する"""
        print(f"=== Generating Collapse Map for Q({self.a},{self.b},{self.c}) [{self.potential_type.upper()}] ===")
        print(f"Target: k = 1 to {max_k}, Length L = 2 to {max_L}\n")
        
        results = {}
        for target_k in range(1, max_k + 1):
            for L in range(2, max_L + 1):
                print(f"Computing H^{{-{target_k}}}_L={L}... ", end="", flush=True)
                if L < target_k + 1:
                    results[(target_k, L)] = 0
                    print("Skipped (Empty space)")
                    continue
                    
                dim_1, _, _, _, v_curr, _ = self._calculate_over_prime(32003, target_k, L)
                if v_curr == 0 or dim_1 == 0:
                    results[(target_k, L)] = 0
                    print("0")
                else:
                    print(f"{dim_1} (Checking GF(31991))... ", end="", flush=True)
                    dim_2, _, _, _, _, _ = self._calculate_over_prime(31991, target_k, L)
                    final_dim = min(dim_1, dim_2)
                    results[(target_k, L)] = final_dim
                    print(f"Confirmed: {final_dim}")

        print(f"\n=== The Collapse Map of Q({self.a},{self.b},{self.c}) [{self.potential_type.upper()}] ===")
        header = f"| Cohomology \\ Length | " + " | ".join([f"L={L}" for L in range(2, max_L + 1)]) + " |"
        print(header)
        print("|---" + "|---" * max_L + "|")
        for target_k in range(1, max_k + 1):
            row = f"| **H^{{-{target_k}}}** | "
            for L in range(2, max_L + 1):
                val = results.get((target_k, L), 0)
                row += f"**{val}** | " if val > 0 else "0 | "
            print(row)
        print("\n")

In [2]:
# 例: Q(2, 2, 2) に対して W = x_0 y_0 z_0 - 3 x_1 y_1 z_1 + 5 x_0 y_1 z_0 を定義
my_W = {
    (0, 0, 0): 1,
    (1, 1, 1): -3,
    (0, 1, 0): 5
}

# custom_W として引き渡す
calc_custom = GinzburgCohomologyCalculator(2, 2, 2, custom_W=my_W)

# ちゃんと読み込まれているか数式で確認
calc_custom.display_potential()
# 出力: W = x_0 y_0 z_0 - 3x_1 y_1 z_1 + 5x_0 y_1 z_0

# コホモロジー計算を実行
calc_custom.compute_single_cohomology(target_k=1, L=3)

W = x_0 y_0 z_0 + 5x_0 y_1 z_0 - 3x_1 y_1 z_1
--- Computing H^{-1} at Length L=3 for Q(2,2,2) [CUSTOM] ---
Dimensions: V_{-0}=24, V_{-1}=24, V_{-2}=3
rank(D_{-1}) = 18
rank(D_{-2}) = 3
--> Double-check Passed! dim H^{-1}_L = 3 (Confirmed)



3

In [3]:
# 2. コラプスマップの生成 
calc_custom.generate_collapse_map(max_k=5, max_L=10)

=== Generating Collapse Map for Q(2,2,2) [CUSTOM] ===
Target: k = 1 to 5, Length L = 2 to 10

Computing H^{-1}_L=2... 0
Computing H^{-1}_L=3... 3 (Checking GF(31991))... Confirmed: 3
Computing H^{-1}_L=4... 6 (Checking GF(31991))... Confirmed: 6
Computing H^{-1}_L=5... 6 (Checking GF(31991))... Confirmed: 6
Computing H^{-1}_L=6... 6 (Checking GF(31991))... Confirmed: 6
Computing H^{-1}_L=7... 6 (Checking GF(31991))... Confirmed: 6
Computing H^{-1}_L=8... 6 (Checking GF(31991))... Confirmed: 6
Computing H^{-1}_L=9... 6 (Checking GF(31991))... Confirmed: 6
Computing H^{-1}_L=10... 6 (Checking GF(31991))... Confirmed: 6
Computing H^{-2}_L=2... Skipped (Empty space)
Computing H^{-2}_L=3... 0
Computing H^{-2}_L=4... 0
Computing H^{-2}_L=5... 0
Computing H^{-2}_L=6... 3 (Checking GF(31991))... Confirmed: 3
Computing H^{-2}_L=7... 6 (Checking GF(31991))... Confirmed: 6
Computing H^{-2}_L=8... 6 (Checking GF(31991))... Confirmed: 6
Computing H^{-2}_L=9... 6 (Checking GF(31991))... Confirmed: 6

In [4]:
# インスタンスを作成
calc = GinzburgCohomologyCalculator(2, 4, 3)

print("=== Alternating ===")
calc.W_coeffs = calc.generate_potential('alternating')
calc.display_potential()

print("\n=== Cyclic Shift ===")
calc.W_coeffs = calc.generate_potential('cyclic_shift')
calc.display_potential()

=== Alternating ===
W = x_0 y_0 z_0 - x_0 y_0 z_1 + x_0 y_0 z_2 - x_0 y_1 z_0 + x_0 y_1 z_1 - x_0 y_1 z_2 + x_0 y_2 z_0 - x_0 y_2 z_1 + x_0 y_2 z_2 - x_0 y_3 z_0 + x_0 y_3 z_1 - x_0 y_3 z_2 - x_1 y_0 z_0 + x_1 y_0 z_1 - x_1 y_0 z_2 + x_1 y_1 z_0 - x_1 y_1 z_1 + x_1 y_1 z_2 - x_1 y_2 z_0 + x_1 y_2 z_1 - x_1 y_2 z_2 + x_1 y_3 z_0 - x_1 y_3 z_1 + x_1 y_3 z_2

=== Cyclic Shift ===
W = - x_0 y_0 z_0 + x_0 y_1 z_1 + x_1 y_0 z_0 - x_1 y_1 z_1


---

Cohomology of $\Gamma_3(Q(1,1,1),W_{gen})$

In [5]:
calc = GinzburgCohomologyCalculator(1, 1, 1)
calc.display_potential()

W = - 70x_0 y_0 z_0


In [6]:
calc.generate_collapse_map(max_k=5, max_L=10)

=== Generating Collapse Map for Q(1,1,1) [GENERIC] ===
Target: k = 1 to 5, Length L = 2 to 10

Computing H^{-1}_L=2... 0
Computing H^{-1}_L=3... 0
Computing H^{-1}_L=4... 0
Computing H^{-1}_L=5... 0
Computing H^{-1}_L=6... 0
Computing H^{-1}_L=7... 0
Computing H^{-1}_L=8... 0
Computing H^{-1}_L=9... 0
Computing H^{-1}_L=10... 0
Computing H^{-2}_L=2... Skipped (Empty space)
Computing H^{-2}_L=3... 0
Computing H^{-2}_L=4... 3 (Checking GF(31991))... Confirmed: 3
Computing H^{-2}_L=5... 3 (Checking GF(31991))... Confirmed: 3
Computing H^{-2}_L=6... 0
Computing H^{-2}_L=7... 0
Computing H^{-2}_L=8... 0
Computing H^{-2}_L=9... 0
Computing H^{-2}_L=10... 0
Computing H^{-3}_L=2... Skipped (Empty space)
Computing H^{-3}_L=3... Skipped (Empty space)
Computing H^{-3}_L=4... 0
Computing H^{-3}_L=5... 0
Computing H^{-3}_L=6... 0
Computing H^{-3}_L=7... 0
Computing H^{-3}_L=8... 0
Computing H^{-3}_L=9... 0
Computing H^{-3}_L=10... 0
Computing H^{-4}_L=2... Skipped (Empty space)
Computing H^{-4}_L=3

Cohomology of $\Gamma_3(Q(1,1,2),W_{gen})$

In [7]:
calc = GinzburgCohomologyCalculator(1, 1, 2)
calc.display_potential()

W = 19x_0 y_0 z_0 - 50x_0 y_0 z_1


In [8]:
calc.generate_collapse_map(max_k=5, max_L=10)

=== Generating Collapse Map for Q(1,1,2) [GENERIC] ===
Target: k = 1 to 5, Length L = 2 to 10

Computing H^{-1}_L=2... 1 (Checking GF(31991))... Confirmed: 1
Computing H^{-1}_L=3... 4 (Checking GF(31991))... Confirmed: 4
Computing H^{-1}_L=4... 3 (Checking GF(31991))... Confirmed: 3
Computing H^{-1}_L=5... 0
Computing H^{-1}_L=6... 0
Computing H^{-1}_L=7... 0
Computing H^{-1}_L=8... 0
Computing H^{-1}_L=9... 0
Computing H^{-1}_L=10... 0
Computing H^{-2}_L=2... Skipped (Empty space)
Computing H^{-2}_L=3... 0
Computing H^{-2}_L=4... 0
Computing H^{-2}_L=5... 3 (Checking GF(31991))... Confirmed: 3
Computing H^{-2}_L=6... 9 (Checking GF(31991))... Confirmed: 9
Computing H^{-2}_L=7... 7 (Checking GF(31991))... Confirmed: 7
Computing H^{-2}_L=8... 2 (Checking GF(31991))... Confirmed: 2
Computing H^{-2}_L=9... 1 (Checking GF(31991))... Confirmed: 1
Computing H^{-2}_L=10... 0
Computing H^{-3}_L=2... Skipped (Empty space)
Computing H^{-3}_L=3... Skipped (Empty space)
Computing H^{-3}_L=4... 0
C

Cohomology of $\Gamma_3(Q(1,1,3),W_{gen})$

In [9]:
calc = GinzburgCohomologyCalculator(1, 1, 3)
calc.display_potential()
calc.generate_collapse_map(max_k=2, max_L=5)

W = 70x_0 y_0 z_0 - 69x_0 y_0 z_1 - 10x_0 y_0 z_2
=== Generating Collapse Map for Q(1,1,3) [GENERIC] ===
Target: k = 1 to 2, Length L = 2 to 5

Computing H^{-1}_L=2... 2 (Checking GF(31991))... Confirmed: 2
Computing H^{-1}_L=3... 12 (Checking GF(31991))... Confirmed: 12
Computing H^{-1}_L=4... 23 (Checking GF(31991))... Confirmed: 23
Computing H^{-1}_L=5... 18 (Checking GF(31991))... Confirmed: 18
Computing H^{-2}_L=2... Skipped (Empty space)
Computing H^{-2}_L=3... 0
Computing H^{-2}_L=4... 0
Computing H^{-2}_L=5... 13 (Checking GF(31991))... Confirmed: 13

=== The Collapse Map of Q(1,1,3) [GENERIC] ===
| Cohomology \ Length | L=2 | L=3 | L=4 | L=5 |
|---|---|---|---|---|---|
| **H^{-1}** | **2** | **12** | **23** | **18** | 
| **H^{-2}** | 0 | 0 | 0 | **13** | 




Cohomology of $\Gamma_3(Q(1,2,2),W_{gen})$

In [10]:
calc = GinzburgCohomologyCalculator(1, 2, 2)
calc.display_potential()
calc.generate_collapse_map(max_k=2, max_L=5)

W = 98x_0 y_0 z_0 - 38x_0 y_0 z_1 - 95x_0 y_1 z_0 + 29x_0 y_1 z_1
=== Generating Collapse Map for Q(1,2,2) [GENERIC] ===
Target: k = 1 to 2, Length L = 2 to 5

Computing H^{-1}_L=2... 0
Computing H^{-1}_L=3... 3 (Checking GF(31991))... Confirmed: 3
Computing H^{-1}_L=4... 8 (Checking GF(31991))... Confirmed: 8
Computing H^{-1}_L=5... 5 (Checking GF(31991))... Confirmed: 5
Computing H^{-2}_L=2... Skipped (Empty space)
Computing H^{-2}_L=3... 0
Computing H^{-2}_L=4... 1 (Checking GF(31991))... Confirmed: 1
Computing H^{-2}_L=5... 4 (Checking GF(31991))... Confirmed: 4

=== The Collapse Map of Q(1,2,2) [GENERIC] ===
| Cohomology \ Length | L=2 | L=3 | L=4 | L=5 |
|---|---|---|---|---|---|
| **H^{-1}** | 0 | **3** | **8** | **5** | 
| **H^{-2}** | 0 | 0 | **1** | **4** | 




Cohomology of $\Gamma_3(Q(1,2,3),W_{gen})$

In [11]:
calc = GinzburgCohomologyCalculator(1, 2, 3)
calc.display_potential()
calc.generate_collapse_map(max_k=2, max_L=5)

W = 94x_0 y_0 z_0 + 20x_0 y_0 z_1 - 39x_0 y_0 z_2 - 67x_0 y_1 z_0 - 75x_0 y_1 z_1 - 90x_0 y_1 z_2
=== Generating Collapse Map for Q(1,2,3) [GENERIC] ===
Target: k = 1 to 2, Length L = 2 to 5

Computing H^{-1}_L=2... 1 (Checking GF(31991))... Confirmed: 1
Computing H^{-1}_L=3... 9 (Checking GF(31991))... Confirmed: 9
Computing H^{-1}_L=4... 25 (Checking GF(31991))... Confirmed: 25
Computing H^{-1}_L=5... 27 (Checking GF(31991))... Confirmed: 27
Computing H^{-2}_L=2... Skipped (Empty space)
Computing H^{-2}_L=3... 0
Computing H^{-2}_L=4... 0
Computing H^{-2}_L=5... 8 (Checking GF(31991))... Confirmed: 8

=== The Collapse Map of Q(1,2,3) [GENERIC] ===
| Cohomology \ Length | L=2 | L=3 | L=4 | L=5 |
|---|---|---|---|---|---|
| **H^{-1}** | **1** | **9** | **25** | **27** | 
| **H^{-2}** | 0 | 0 | 0 | **8** | 




Cohomology of $\Gamma_3(Q(1,3,3),W_{gen})$

In [12]:
calc = GinzburgCohomologyCalculator(1, 3, 3)
calc.display_potential()
calc.generate_collapse_map(max_k=2, max_L=5)

W = - 23x_0 y_0 z_0 - 9x_0 y_0 z_1 - 70x_0 y_0 z_2 + 15x_0 y_1 z_0 + 100x_0 y_1 z_1 - 49x_0 y_1 z_2 + 34x_0 y_2 z_0 - 68x_0 y_2 z_1 + 14x_0 y_2 z_2
=== Generating Collapse Map for Q(1,3,3) [GENERIC] ===
Target: k = 1 to 2, Length L = 2 to 5

Computing H^{-1}_L=2... 0
Computing H^{-1}_L=3... 8 (Checking GF(31991))... Confirmed: 8
Computing H^{-1}_L=4... 42 (Checking GF(31991))... Confirmed: 42
Computing H^{-1}_L=5... 55 (Checking GF(31991))... Confirmed: 55
Computing H^{-2}_L=2... Skipped (Empty space)
Computing H^{-2}_L=3... 0
Computing H^{-2}_L=4... 1 (Checking GF(31991))... Confirmed: 1
Computing H^{-2}_L=5... 6 (Checking GF(31991))... Confirmed: 6

=== The Collapse Map of Q(1,3,3) [GENERIC] ===
| Cohomology \ Length | L=2 | L=3 | L=4 | L=5 |
|---|---|---|---|---|---|
| **H^{-1}** | 0 | **8** | **42** | **55** | 
| **H^{-2}** | 0 | 0 | **1** | **6** | 




Cohomology of $\Gamma_3(Q(2,2,2),W_{gen})$

In [13]:
calc = GinzburgCohomologyCalculator(2, 2, 2)
calc.display_potential()
calc.generate_collapse_map(max_k=2, max_L=5)

W = 91x_0 y_0 z_0 + 83x_0 y_0 z_1 - 6x_0 y_1 z_0 - 49x_0 y_1 z_1 - 11x_1 y_0 z_0 + 4x_1 y_0 z_1 - 30x_1 y_1 z_0 + 70x_1 y_1 z_1
=== Generating Collapse Map for Q(2,2,2) [GENERIC] ===
Target: k = 1 to 2, Length L = 2 to 5

Computing H^{-1}_L=2... 0
Computing H^{-1}_L=3... 3 (Checking GF(31991))... Confirmed: 3
Computing H^{-1}_L=4... 6 (Checking GF(31991))... Confirmed: 6
Computing H^{-1}_L=5... 6 (Checking GF(31991))... Confirmed: 6
Computing H^{-2}_L=2... Skipped (Empty space)
Computing H^{-2}_L=3... 0
Computing H^{-2}_L=4... 0
Computing H^{-2}_L=5... 0

=== The Collapse Map of Q(2,2,2) [GENERIC] ===
| Cohomology \ Length | L=2 | L=3 | L=4 | L=5 |
|---|---|---|---|---|---|
| **H^{-1}** | 0 | **3** | **6** | **6** | 
| **H^{-2}** | 0 | 0 | 0 | 0 | 




Cohomology of $\Gamma_3(Q(2,2,3),W_{gen})$

In [14]:
calc = GinzburgCohomologyCalculator(2, 2, 3)
calc.display_potential()
calc.generate_collapse_map(max_k=2, max_L=5)

W = - 48x_0 y_0 z_0 + 67x_0 y_0 z_1 - 94x_0 y_0 z_2 + 88x_0 y_1 z_0 - 76x_0 y_1 z_1 + 27x_0 y_1 z_2 + 15x_1 y_0 z_0 + 67x_1 y_0 z_1 + 27x_1 y_0 z_2 - 73x_1 y_1 z_0 + 5x_1 y_1 z_1 - 55x_1 y_1 z_2
=== Generating Collapse Map for Q(2,2,3) [GENERIC] ===
Target: k = 1 to 2, Length L = 2 to 5

Computing H^{-1}_L=2... 0
Computing H^{-1}_L=3... 0
Computing H^{-1}_L=4... 5 (Checking GF(31991))... Confirmed: 5
Computing H^{-1}_L=5... 12 (Checking GF(31991))... Confirmed: 12
Computing H^{-2}_L=2... Skipped (Empty space)
Computing H^{-2}_L=3... 0
Computing H^{-2}_L=4... 0
Computing H^{-2}_L=5... 1 (Checking GF(31991))... Confirmed: 1

=== The Collapse Map of Q(2,2,3) [GENERIC] ===
| Cohomology \ Length | L=2 | L=3 | L=4 | L=5 |
|---|---|---|---|---|---|
| **H^{-1}** | 0 | 0 | **5** | **12** | 
| **H^{-2}** | 0 | 0 | 0 | **1** | 




Cohomology of $\Gamma_3(Q(2,3,3),W_{gen})$

In [15]:
calc = GinzburgCohomologyCalculator(2, 3, 3)
calc.display_potential()
calc.generate_collapse_map(max_k=2, max_L=5)

W = - 62x_0 y_0 z_0 + 64x_0 y_0 z_1 + 97x_0 y_0 z_2 - 35x_0 y_1 z_0 - 98x_0 y_1 z_1 + 61x_0 y_1 z_2 + 40x_0 y_2 z_0 + 83x_0 y_2 z_1 + 46x_0 y_2 z_2 + 90x_1 y_0 z_0 + 58x_1 y_0 z_1 + 63x_1 y_0 z_2 + 89x_1 y_1 z_0 - 96x_1 y_1 z_1 - 74x_1 y_1 z_2 + 92x_1 y_2 z_0 - 96x_1 y_2 z_1 + 49x_1 y_2 z_2
=== Generating Collapse Map for Q(2,3,3) [GENERIC] ===
Target: k = 1 to 2, Length L = 2 to 5

Computing H^{-1}_L=2... 0
Computing H^{-1}_L=3... 2 (Checking GF(31991))... Confirmed: 2
Computing H^{-1}_L=4... 13 (Checking GF(31991))... Confirmed: 13
Computing H^{-1}_L=5... 29 (Checking GF(31991))... Confirmed: 29
Computing H^{-2}_L=2... Skipped (Empty space)
Computing H^{-2}_L=3... 0
Computing H^{-2}_L=4... 0
Computing H^{-2}_L=5... 0

=== The Collapse Map of Q(2,3,3) [GENERIC] ===
| Cohomology \ Length | L=2 | L=3 | L=4 | L=5 |
|---|---|---|---|---|---|
| **H^{-1}** | 0 | **2** | **13** | **29** | 
| **H^{-2}** | 0 | 0 | 0 | 0 | 




Cohomology of $\Gamma_3(Q(3,3,3),W_{gen})$

In [16]:
calc = GinzburgCohomologyCalculator(3, 3, 3)
calc.display_potential()

W = 31x_0 y_0 z_0 + 81x_0 y_0 z_1 + 77x_0 y_0 z_2 - 22x_0 y_1 z_0 + 53x_0 y_1 z_1 + 26x_0 y_1 z_2 - 90x_0 y_2 z_0 + 82x_0 y_2 z_1 - 15x_0 y_2 z_2 + 73x_1 y_0 z_0 + 3x_1 y_0 z_1 + 41x_1 y_0 z_2 + 51x_1 y_1 z_0 - 88x_1 y_1 z_1 - 44x_1 y_1 z_2 + 20x_1 y_2 z_0 - 81x_1 y_2 z_1 - 84x_1 y_2 z_2 - 81x_2 y_0 z_0 + 25x_2 y_0 z_1 + 70x_2 y_0 z_2 - 8x_2 y_1 z_0 + 90x_2 y_1 z_1 - 20x_2 y_1 z_2 + 90x_2 y_2 z_0 - 18x_2 y_2 z_1 - 19x_2 y_2 z_2


In [17]:
calc.generate_collapse_map(max_k=3, max_L=7)

=== Generating Collapse Map for Q(3,3,3) [GENERIC] ===
Target: k = 1 to 3, Length L = 2 to 7

Computing H^{-1}_L=2... 0
Computing H^{-1}_L=3... 0
Computing H^{-1}_L=4... 0
Computing H^{-1}_L=5... 0
Computing H^{-1}_L=6... 0
Computing H^{-1}_L=7... 0
Computing H^{-2}_L=2... Skipped (Empty space)
Computing H^{-2}_L=3... 0
Computing H^{-2}_L=4... 0
Computing H^{-2}_L=5... 0
Computing H^{-2}_L=6... 0
Computing H^{-2}_L=7... 0
Computing H^{-3}_L=2... Skipped (Empty space)
Computing H^{-3}_L=3... Skipped (Empty space)
Computing H^{-3}_L=4... 0
Computing H^{-3}_L=5... 0
Computing H^{-3}_L=6... 0
Computing H^{-3}_L=7... 0

=== The Collapse Map of Q(3,3,3) [GENERIC] ===
| Cohomology \ Length | L=2 | L=3 | L=4 | L=5 | L=6 | L=7 |
|---|---|---|---|---|---|---|---|
| **H^{-1}** | 0 | 0 | 0 | 0 | 0 | 0 | 
| **H^{-2}** | 0 | 0 | 0 | 0 | 0 | 0 | 
| **H^{-3}** | 0 | 0 | 0 | 0 | 0 | 0 | 


